# Cheat Sheet — GDP Growth Prediction (Extended Project)

Quick-reference syntax for every technique used in this project. Snippets use small dummy data so every cell runs standalone — copy the *pattern* into your working notebook.

Sections: Setup · Inspection · Accounting-Style Negative Numbers · Whitespace/Casing Cleanup · Two Kinds of Missingness · Ordinal Encoding & Interactions · EDA & Leakage Checks · Encoding · Models · Tuning & CV · Evaluation · Feature Importance · Scenario Simulation · Persistence.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 1. Loading & Inspection

In [ ]:
df = pd.read_csv('economy_indicators.csv')
df.head()
df.shape
df.info()
df.describe().T


## 2. Accounting-Style Negative Numbers: Parentheses, Not a Minus Sign

In [ ]:
# "4.2%" -> 4.2   |   "(1.7%)" -> -1.7
def parse_pct(series):
    s = series.astype(str).str.strip()
    is_negative = s.str.startswith('(')
    numeric_part = (s.str.replace('(', '', regex=False)
                      .str.replace(')', '', regex=False)
                      .str.replace('%', '', regex=False)
                      .astype(float))
    return np.where(is_negative, -numeric_part, numeric_part)

demo = pd.Series(['4.2%', '(1.7%)', '0.0%', '(10.5%)'])
print(parse_pct(demo))   # [ 4.2 -1.7  0.  -10.5]

# This convention shows up constantly in real financial/economic exports (Excel's
# "accounting" number format renders negatives in parentheses) -- always check a
# sample of values before assuming a percentage column is minus-sign-only.


## 3. Whitespace & Casing Cleanup for Categorical Identifiers

In [ ]:
messy = pd.Series(['Costa Rica', ' Costa Rica ', 'COSTA RICA', 'costa  rica', 'Costa  Rica'])
cleaned = messy.str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
print(cleaned.unique())   # collapses down to a single 'Costa Rica'

# .str.strip()              -> removes leading/trailing whitespace
# .str.replace(r'\s+',' ')  -> collapses any run of internal whitespace to one space
# .str.title()               -> normalizes casing
# Always run .nunique() before AND after to see how much this cleanup actually mattered.


## 4. Two Kinds of Missing Values

In [ ]:
# TYPE 1: random reporting gap -- genuinely unknown, needs a flag + impute
raw = pd.Series(['3.10 million visitors', '..', '1.85 million visitors'])
is_missing = raw == '..'
flag = is_missing.astype(int)
cleaned = raw.str.replace(' million visitors', '').replace('..', np.nan).astype(float)
imputed = cleaned.fillna(cleaned.median())

# TYPE 2: structural absence -- the value doesn't exist BECAUSE of another column's state,
# not because it's unknown. Impute with the domain-correct constant, no flag needed
# (the OTHER column already tells you why it's "missing").
disaster_flag = pd.Series([1, 0, 1])          # 1 = disaster happened
damage_raw = pd.Series(['5.2%', '..', '1.0%']) # damage only recorded WHEN disaster happened
damage_is_missing = damage_raw == '..'
damage_cleaned = damage_raw.replace('..', np.nan)
damage_cleaned = pd.to_numeric(damage_cleaned.str.replace('%', ''), errors='coerce')
damage_final = damage_cleaned.fillna(0)   # no disaster -> true damage IS zero, not "unknown"

# Rule of thumb: ask "if I knew everything about this row except this value, could I
# DEDUCE the missing value with certainty?" If yes (as with damage given no disaster),
# it's structural -- impute the deduced constant. If no, it's a genuine reporting gap --
# flag it and impute a statistical estimate (median/mean).


## 5. Ordinal Encoding & Hand-Crafted Interaction Features

In [ ]:
income_map = {'Low income': 0, 'Lower middle income': 1, 'Upper middle income': 2, 'High income': 3}
df_demo = pd.DataFrame({'income_group': ['Upper middle income', 'High income']})
df_demo['income_group_score'] = df_demo['income_group'].map(income_map)

# A linear model has NO way to represent "the effect of A depends on the level of B"
# unless you hand it the product term directly:
# df['disaster_debt_interaction'] = df['natural_disaster_event'] * df['government_debt_pct_gdp']
#
# A tree model CAN represent this via nested splits, but reliably discovering a rare
# interaction (e.g. present in only ~10% of rows) from splits alone often needs more
# data than a small dataset provides -- handing it the pre-multiplied feature helps
# both model families, and is ESSENTIAL for a linear model to use it at all.


## 6. EDA & the Forecast-Leakage Check

In [ ]:
numeric_df = df.select_dtypes(include='number')
numeric_df.corr()['gdp_growth_rate_pct'].sort_values(ascending=False)

df['gdp_growth_rate_pct'].corr(df['imf_next_year_growth_forecast_pct'])
# Even a MODERATE correlation (e.g. ~0.9, lower than prior projects' 0.95+ traps) can still
# be leakage relative to your goal -- the magnitude of the correlation and the strength of
# the leakage argument are two separate questions. Ask what the column REPRESENTS
# (an expert's own prediction for the same target) before deciding from correlation alone.

from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = numeric_df.drop(columns=['gdp_growth_rate_pct']).dropna()
vif = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)


## 7. Leakage-Safe Target Encoding

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
low_card = [c for c in cat_cols if df[c].nunique() < 5]     # region, quarter, currency_regime
high_card = [c for c in cat_cols if df[c].nunique() >= 5]   # country, primary_export_sector

df_enc = pd.get_dummies(df, columns=low_card, drop_first=True)
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

from sklearn.model_selection import train_test_split
X = df_enc.drop(columns=['gdp_growth_rate_pct'])   # + drop the IMF forecast column too
y = df_enc['gdp_growth_rate_pct']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

global_mean = y_train.mean()
for col in high_card:
    means = y_train.groupby(X_train[col]).mean()
    X_train[col] = X_train[col].map(means)
    X_test[col] = X_test[col].map(means).fillna(global_mean)


## 8. Models

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

lin = LinearRegression().fit(X_train, y_train)                 # baseline linear
ridge = Ridge(alpha=1.0, random_state=RANDOM_STATE).fit(X_train, y_train)
rf = RandomForestRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
gb = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)


## 9. Cross-Validation & Tuning

In [ ]:
from sklearn.model_selection import cross_val_score, RandomizedSearchCV

scores = -cross_val_score(ridge, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(scores.mean(), scores.std())

param_dist = {'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]}
search = RandomizedSearchCV(Ridge(random_state=RANDOM_STATE), param_distributions=param_dist,
                             n_iter=7, cv=5, scoring='neg_mean_absolute_error',
                             random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)
best_model = search.best_estimator_


## 10. Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
# CAUTION: MAPE is unstable when growth is near 0% for a given quarter -- same caveat as
# the sports-betting project's near-zero-spread problem.
safe_denom = y_test.replace(0, np.nan)
mape = np.nanmean(np.abs((y_test - y_pred) / safe_denom)) * 100

plt.scatter(y_test, y_pred, alpha=0.5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('Actual growth (%)'); plt.ylabel('Predicted growth (%)')
plt.show()


## 11. Feature Importance

In [ ]:
coefs = pd.Series(best_model.coef_, index=X_train.columns).sort_values(key=abs, ascending=False)
sns.barplot(x=coefs.head(10).values, y=coefs.head(10).index)
plt.show()

from sklearn.inspection import permutation_importance
result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_idx = result.importances_mean.argsort()[-10:][::-1]
sns.barplot(x=result.importances_mean[perm_idx], y=X_test.columns[perm_idx])
plt.show()


## 12. Policy Scenario Simulation Pattern

In [ ]:
def simulate_scenario(model, base_row, changes: dict):
    """base_row: a single-row DataFrame in the model's exact input format.
    changes: {column_name: new_value} overrides to apply before predicting.
    Returns (baseline_prediction, scenario_prediction, delta).
    """
    baseline_pred = model.predict(base_row)[0]
    scenario_row = base_row.copy()
    for col, val in changes.items():
        scenario_row[col] = val
    # if the interaction term depends on a changed column, recompute it here, e.g.:
    if 'natural_disaster_event' in changes or 'government_debt_pct_gdp' in changes:
        scenario_row['disaster_debt_interaction'] = (
            scenario_row['natural_disaster_event'] * scenario_row['government_debt_pct_gdp']
        )
    scenario_pred = model.predict(scenario_row)[0]
    return baseline_pred, scenario_pred, scenario_pred - baseline_pred

# Example: a 15% commodity price shock
# base_row = X_test.loc[[some_index]]
# baseline, scenario, delta = simulate_scenario(
#     best_model, base_row, {'commodity_export_price_index': base_row['commodity_export_price_index'].iloc[0] * 0.85}
# )
# print(f"Baseline: {baseline:.2f}%  Scenario: {scenario:.2f}%  Impact: {delta:+.2f} pp")


## 13. Persistence & Inference

In [ ]:
import joblib

joblib.dump(best_model, 'growth_model.pkl')
joblib.dump({'target_encoding_maps': {}, 'model_columns': list(X_train.columns),
             'income_map': income_map}, 'growth_encoders.pkl')

def predict_growth(raw_dict, model, encoders):
    # 1. put raw_dict into a one-row DataFrame
    # 2. re-apply the SAME cleaning (accounting-pct parsing, whitespace/casing, missingness
    #    handling, ordinal + interaction features, encoding) used in training
    # 3. reindex to training column order, filling missing dummy columns with 0
    # 4. return model.predict(row)[0]
    pass


## Quick lookup: two kinds of missingness

| Question | Random reporting gap | Structural absence |
|---|---|---|
| Can another column tell you the true value with certainty? | No | Yes |
| Example | `tourism_arrivals` unreported for a small economy | `disaster_damage_pct_gdp` when there was no disaster |
| Imputation | Flag column + median/mean | Domain-correct constant (often 0), no flag needed |
